In [1]:
import pandas as pd
import numpy as np

In [2]:
A_df = pd.read_csv("data/A.csv", header=None)
B_df = pd.read_csv("data/B.csv", header=None)
C_df = pd.read_csv("data/C.csv", header=None)

# Convert all string-looking numbers to floats
A = A_df.apply(pd.to_numeric, errors='coerce').values
B = B_df.apply(pd.to_numeric, errors='coerce').values
C = C_df.apply(pd.to_numeric, errors='coerce').values

In [3]:
A_index_df = pd.read_csv("data/index_A.csv")
B_index_df = pd.read_csv("data/index_B.csv")
C_index_df = pd.read_csv("data/index_C.csv")

## Remove Transporation

In [4]:
A_transport_df = pd.read_csv("data/Transportation_A.csv")

In [5]:
# create a dict mapping each provider name to all its indices in A_index_df
mapping = A_index_df.groupby('provider name')['index'].apply(list)

# build a single flat list of all matching indices for the foreground processes
matched_indices_transport = [
    idx
    for name in A_transport_df['provider name']
    if name in mapping
    for idx in mapping[name]
]

In [6]:
import numpy as np

# matched_indices_transport is the list of indices to remove
to_drop = np.array(sorted(set(matched_indices_transport), key=int))

# 1) Remove from A_index_df
mask_keep = ~A_index_df['index'].isin(to_drop)
A_index_df = A_index_df.loc[mask_keep].copy()

# 2) Remove corresponding rows and columns from A
A = np.delete(A, to_drop, axis=0)  # remove rows
A = np.delete(A, to_drop, axis=1)  # remove columns

# 3) Remove the same columns from B (keep rows)
B = np.delete(B, to_drop, axis=1)

# 4) Reset the index column in A_index_df
A_index_df['index'] = np.arange(len(A_index_df), dtype=int)

## Remove and aggregate Electricity

In [7]:
A_elec_df = pd.read_csv("data/Electricity_A.csv")

In [8]:
# Inputs assumed:
# A : numeric numpy array (rows x cols)
# A_index_df : DataFrame with columns ["index", "provider name", "flow name", ...]
# A_elec_df : DataFrame with column ["provider name"] listing all electricity providers
# The indices in A_index_df["index"] align with both row and column positions of A.

# 0) Build the set of electricity provider names
elec_names = set(A_elec_df['provider name'].dropna().astype(str).unique())

# 1) Find their indices in A_index_df
elec_idx = A_index_df.loc[A_index_df['provider name'].astype(str).isin(elec_names), 'index'].astype(int).unique()

# 2) Locate the mix row index (must exist)
mix_name = "Electricity Mix (Global)"
mix_rows = A_index_df.loc[A_index_df['provider name'] == mix_name, 'index'].astype(int).unique()
if len(mix_rows) == 0:
    raise ValueError("Electricity Mix (Global) not found in A_index_df['provider name'].")
mix_idx = int(mix_rows[0])

# Ensure the mix row is not purged
elec_idx_set = set(map(int, elec_idx))
elec_idx_wo_mix = sorted(elec_idx_set - {mix_idx})

# 3) Aggregate: add all electricity rows (except the mix row) into the mix row, column-wise
if len(elec_idx_wo_mix) > 0:
    # in case of NaNs
    add_block = np.nansum(A[elec_idx_wo_mix, :], axis=0)
    A[mix_idx, :] = np.nan_to_num(A[mix_idx, :]) + np.nan_to_num(add_block)

# 4) Decide what to drop
rows_to_drop = np.array(elec_idx_wo_mix, dtype=int)            # drop electricity rows except the mix row
cols_to_drop = np.array(elec_idx_wo_mix, dtype=int)            # drop electricity columns except the mix column

# (Optionally also drop the mix COLUMN; keep it if you want to retain that process as a column)
# To ALSO drop the mix column, uncomment the next line:
# cols_to_drop = np.array(sorted(elec_idx_set), dtype=int)

# 5) Remove rows/columns from A and columns from B
if rows_to_drop.size > 0:
    A = np.delete(A, rows_to_drop, axis=0)
if cols_to_drop.size > 0:
    A = np.delete(A, cols_to_drop, axis=1)
    B = np.delete(B, cols_to_drop, axis=1)

# 6) Remove the same rows from A_index_df (only rows; columns in A_index_df are metadata)
if len(elec_idx_wo_mix) > 0:
    keep_mask = ~A_index_df['index'].astype(int).isin(elec_idx_wo_mix)
    A_index_df = A_index_df.loc[keep_mask].copy()

# 7) Reset the "index" column in A_index_df to reflect 0..n-1 after deletions
A_index_df['index'] = np.arange(len(A_index_df), dtype=int)

In [9]:
# 8) Zero specific entries in A for the mix column
mix_name = "Electricity Mix (Global)"
mix_col_idx_s = A_index_df.loc[A_index_df['provider name'] == mix_name, 'index'].astype(int)

if mix_col_idx_s.empty:
    raise ValueError("Mix column not found after pruning. Did you drop the mix column?")
mix_col = int(mix_col_idx_s.iloc[0])

target_names = {"Fossil Electricity", "Clean Electricity"}
row_idxs = (
    A_index_df.loc[A_index_df['provider name'].isin(target_names), 'index']
    .astype(int)
    .to_numpy()
)

if row_idxs.size > 0:
    A[row_idxs, mix_col] = 0.0
else:
    print("Warning: no rows named 'Fossil Electricity' or 'Clean Electricity' found after pruning.")

## Quality Scenario (for recycled material from mechanical)

In [10]:
quality_scenarios_df = pd.read_csv("data/quality_scenarios.csv")

In [11]:
# Assumes the following are already in memory:
# - quality_scenarios_df  with columns: "Recycling Process", "Substitutable Virgin Process", "S1 - no limit"
# - A_index_df            with columns: "Provider name", "Index"
# - A                     as a NumPy array (your A-matrix)

# Normalize helper (case-insensitive, trim spaces)
_norm = lambda s: str(s).strip().casefold()

# Build name -> index map from A_index_df
# If A_index_df['Index'] is 1-based, uncomment the "- 1" line below instead.
name_to_idx = {
    _norm(p): int(i)
    for p, i in zip(A_index_df["provider name"], A_index_df["index"])
    # for p, i in zip(A_index_df["Provider name"], A_index_df["Index"] - 1)  # <- use this if indices are 1-based
}

# Map processes to A indices
col_idx = quality_scenarios_df["Recycling Process"].astype(str).map(_norm).map(name_to_idx)
row_idx = quality_scenarios_df["Substitutable Virgin Process"].astype(str).map(_norm).map(name_to_idx)

# Save the pair (row, col) to the "Index" column
quality_scenarios_df["index"] = list(zip(row_idx, col_idx))

# Pull values to write
vals = pd.to_numeric(quality_scenarios_df["S1 - no limit"], errors="coerce")

# Only update where both indices and value are valid
mask = row_idx.notna() & col_idx.notna() & vals.notna()
rows = row_idx[mask].astype(int).to_numpy()
cols = col_idx[mask].astype(int).to_numpy()
v    = vals[mask].to_numpy(dtype=float)

# Write into A at (row, col)
A[rows, cols] = v

# Optional diagnostics:
# print("Updated entries:", mask.sum())
# print("Unmatched Recycling Process:", quality_scenarios_df.loc[col_idx.isna(), "Recycling Process"].drop_duplicates().tolist()[:10])
# print("Unmatched Substitutable Virgin Process:", quality_scenarios_df.loc[row_idx.isna(), "Substitutable Virgin Process"].drop_duplicates().tolist()[:10])

## Setting up A design

In [12]:
import numpy as np
import pandas as pd

# --- VALIDATION ---
if not isinstance(A, np.ndarray) or A.ndim != 2:
    raise ValueError("A must be a 2D NumPy array.")
n, m = A.shape
if n != m:
    raise ValueError(f"A must be square; got {A.shape}.")
need = {"index", "provider name", "flow name"}
if not need.issubset(A_index_df.columns):
    raise ValueError(f"A_index_df missing columns: {need - set(A_index_df.columns)}")

# ensure indices cover 0..n-1 uniquely
idx = pd.to_numeric(A_index_df["index"], errors="coerce").astype("Int64")
if idx.isna().any():
    raise ValueError("A_index_df['index'] contains non-integer values.")
idx_vals = idx.astype(int).to_numpy()
if len(np.unique(idx_vals)) != len(idx_vals):
    dupes = A_index_df[A_index_df.duplicated("index", keep=False)].sort_values("index")
    raise ValueError(f"Duplicate indices in A_index_df['index']:\n{dupes}")
if idx_vals.min() != 0 or idx_vals.max() != n - 1:
    raise ValueError(f"A_index_df['index'] must span 0..{n-1}.")

# --- build label arrays aligned by index ---
providers = [""] * n
flows     = [""] * n
for _, r in A_index_df.iterrows():
    i = int(r["index"])
    providers[i] = str(r["provider name"])
    flows[i]     = str(r["flow name"])

# --- assemble (no titles, no extra columns) ---
rows = []
# first two header rows (blank first two cells, then provider and flow names)
rows.append(["", ""] + providers)
rows.append(["", ""] + flows)

# numeric block with matching row labels
for i in range(n):
    rows.append([providers[i], flows[i]] + A[i, :].tolist())

final_df = pd.DataFrame(rows)

# optional: export
# final_df.to_csv("A_full_labeled_aligned_clean.csv", index=False, header=False)
A_design_df = final_df

In [13]:
decision_variables_all = pd.read_csv("data/decision_variables_dynamic.csv")
new_flow_df = pd.read_csv("data/new_flow_roadmap.csv")

In [14]:
import pandas as pd
import numpy as np

def apply_levers_add_and_populate(
    A_design_df: pd.DataFrame,
    decision_variables_all: pd.DataFrame,
    new_flow_df: pd.DataFrame,
    activate: list,
    case_insensitive: bool = False,
    overwrite: bool = True,
):
    """
    Full pipeline:
      1) Remove rows from A_design_df where 'Input parameters names' has ANY activated lever == 1
         (match against the FIRST COLUMN of A_design_df; header rows 0,1 are preserved)
      2) From removed rows, capture UNIQUE PROVIDER NAMES (col 0)
      3) Map providers via new_flow_df (Process -> Flow name), dedupe flows, append blank rows:
            ["", <Flow name>, 0, 0, ...]
      4) Populate those new rows: for each matching Flow name in new_flow_df,
         write -1 (input) or +1 (output) at the column whose header (row 0) equals 'Process'.

    Expected layout for A_design_df:
      - Row 0: ['', ''] + provider headers across (cols 2..end)
      - Row 1: ['', ''] + flow headers across    (cols 2..end)
      - Row 2+: [provider_name, flow_name] + numeric (cols 2..end)

    Returns
    -------
    updated_df : pd.DataFrame
    report : dict with counts/diagnostics
    """

    # --------- validation ---------
    if A_design_df.shape[0] < 2 or A_design_df.shape[1] < 3:
        raise ValueError("A_design_df must have 2 header rows and >=3 columns.")
    if "Input parameters names" not in decision_variables_all.columns:
        raise ValueError("decision_variables_all must include 'Input parameters names'.")
    need_nf = {"Flow name", "Process", "Type of flow"}
    if not need_nf.issubset(new_flow_df.columns):
        raise ValueError(f"new_flow_df must include columns: {need_nf}")

    df  = A_design_df.copy()
    dva = decision_variables_all.copy()
    nf  = new_flow_df.copy()

    # --------- step 1: removal (by active levers) ---------
    lever_cols = [c for c in activate if c in dva.columns]
    if not lever_cols:
        raise ValueError(f"No requested levers found in decision_variables_all. Requested: {activate}")

    for c in lever_cols:
        dva[c] = pd.to_numeric(dva[c], errors="coerce").fillna(0).astype(int)

    # Key choice for matching (exact vs case-insensitive)
    if case_insensitive:
        norm = lambda s: str(s).strip().casefold()
        dva["_name"] = dva["Input parameters names"].map(norm)
        first_col    = df.iloc[:, 0].astype(str).map(norm)
    else:
        dva["_name"] = dva["Input parameters names"].astype(str)
        first_col    = df.iloc[:, 0].astype(str)

    to_remove_names = set(dva.loc[dva[lever_cols].sum(axis=1) > 0, "_name"])

    is_header = df.index.isin([0, 1])
    remove_mask = (~is_header) & (first_col.isin(to_remove_names))

    # capture removed providers (first column) & rows count
    removed_rows_count = int(remove_mask.sum())
    removed_providers = (
        df.loc[remove_mask, df.columns[0]].dropna().astype(str).unique().tolist()
    )
    removed_providers = sorted(set(removed_providers))

    # keep flow names too, if you want them for debugging
    removed_flows_dbg = (
        df.loc[remove_mask, df.columns[1]].dropna().astype(str).unique().tolist()
    )

    filtered_df = df.loc[~remove_mask].reset_index(drop=True)

    # --------- step 2: provider -> flow mapping, add blank rows ---------
    if case_insensitive:
        norm = lambda s: str(s).strip().casefold()
        provider_keys = set(map(norm, removed_providers))
        nf["_proc"] = nf["Process"].astype(str).map(norm)
        flows_to_add = nf.loc[nf["_proc"].isin(provider_keys), "Flow name"].astype(str).unique().tolist()
    else:
        provider_keys = set(map(str, removed_providers))
        flows_to_add = nf.loc[nf["Process"].astype(str).isin(provider_keys), "Flow name"].astype(str).unique().tolist()

    flows_to_add = sorted(set(flows_to_add))

    numeric_len = filtered_df.shape[1] - 2
    zero_row = [0.0] * numeric_len
    new_rows = [["", flow] + zero_row for flow in flows_to_add]

    if new_rows:
        add_df = pd.DataFrame(new_rows, columns=filtered_df.columns)
        with_added_df = pd.concat([filtered_df, add_df], ignore_index=True)
    else:
        with_added_df = filtered_df.copy()

    # --------- step 3: populate the newly added rows ---------
    # Build header map from row 0
    if case_insensitive:
        norm = lambda s: str(s).strip().casefold()
        headers_pretty = with_added_df.iloc[0, 2:].astype(str).tolist()
        header_keys    = [norm(h) for h in headers_pretty]
        header_to_col  = {k: 2 + i for i, k in enumerate(header_keys)}
        nf["_flow_key"] = nf["Flow name"].astype(str).map(norm)
        nf["_proc_key"] = nf["Process"].astype(str).map(norm)
        flow_key = lambda x: norm(str(x))
        proc_key = lambda x: norm(str(x))
        proc_display = lambda r: str(r["Process"])
    else:
        headers_pretty = with_added_df.iloc[0, 2:].astype(str).tolist()
        header_to_col  = {h: 2 + i for i, h in enumerate(headers_pretty)}
        nf["_flow_key"] = nf["Flow name"].astype(str)
        nf["_proc_key"] = nf["Process"].astype(str)
        flow_key = lambda x: str(x)
        proc_key = lambda x: str(x)
        proc_display = lambda r: str(r["Process"])

    # mark the newly added rows: provider label blank & index >= 2
    first_col_after = with_added_df.iloc[:, 0].astype(str)
    is_added_row = (with_added_df.index >= 2) & (first_col_after.str.strip() == "")
    target_idxs = with_added_df.index[is_added_row].tolist()

    rows_populated = 0
    skipped_no_matches = []  # flows with no new_flow_df rows
    skipped_no_column  = []  # (process, flow) when process header not found
    skipped_bad_type   = []  # (process, flow, type)

    for i in target_idxs:
        flow_label = str(with_added_df.iat[i, 1])
        if not flow_label:
            continue

        sub = nf.loc[nf["_flow_key"] == flow_key(flow_label)]
        if sub.empty:
            skipped_no_matches.append(flow_label)
            continue

        wrote_any = False
        for _, r in sub.iterrows():
            typ = str(r["Type of flow"]).strip().casefold()
            if typ == "input":
                val = +1.0
            elif typ == "output":
                val = -1.0
            else:
                skipped_bad_type.append((proc_display(r), flow_label, r["Type of flow"]))
                continue

            # find target column (row-0 header == Process)
            col_j = header_to_col.get(proc_key(r["Process"])) if case_insensitive else header_to_col.get(r["Process"])
            if col_j is None:
                skipped_no_column.append((proc_display(r), flow_label))
                continue

            current = with_added_df.iat[i, col_j]
            if overwrite or (pd.isna(current) or float(current) == 0.0):
                with_added_df.iat[i, col_j] = val
                wrote_any = True

        if wrote_any:
            rows_populated += 1

    updated_df = with_added_df.reset_index(drop=True)

    # --------- report ---------
    report = {
        "requested_levers": lever_cols,
        "removed_rows_count": removed_rows_count,
        "removed_providers_unique": len(removed_providers),
        "removed_providers": removed_providers,
        "removed_flows_dbg": removed_flows_dbg,
        "flows_to_add": flows_to_add,
        "rows_added": len(flows_to_add),
        "rows_populated": rows_populated,
        "skipped_no_matches": skipped_no_matches,
        "skipped_no_column": skipped_no_column,
        "skipped_bad_type": skipped_bad_type,
    }
    return updated_df, report


def apply_levers_and_get_matrix(
    A_design_df: pd.DataFrame,
    decision_variables_all: pd.DataFrame,
    new_flow_df: pd.DataFrame,
    activate: list,
    case_insensitive: bool = False,
    overwrite: bool = True,
):
    """
    Wrapper that:
      - runs apply_levers_add_and_populate
      - extracts the numeric submatrix from row 3, col 3 onward
      - returns: updated_df, report, numeric_subdf, numeric_matrix (np.ndarray)
    """
    updated_df, report = apply_levers_add_and_populate(
        A_design_df=A_design_df,
        decision_variables_all=decision_variables_all,
        new_flow_df=new_flow_df,
        activate=activate,
        case_insensitive=case_insensitive,
        overwrite=overwrite,
    )

    # Extract matrix from 3rd row and 3rd column onward, coerce to numeric
    A_raw = updated_df.iloc[2:, 2:]
    A_numeric = A_raw.apply(pd.to_numeric, errors="coerce")
    A_matrix = A_numeric.to_numpy()

    return updated_df, report, A_numeric, A_matrix

In [15]:
# ------------------ example call ------------------
# Options:
# Circularity
# Upstream Chemicals
# Electricity Source
# Carbon Capture
# Microplastic Treatment
# All

activated = ["All"]

A_design_final_df, rep, A_numeric_df, A_design_matrix = apply_levers_and_get_matrix(
    A_design_df,
    decision_variables_all,
    new_flow_df,
    activate=activated,
    case_insensitive=True,  # robust against case/space differences
    overwrite=True          # set to False to only fill zeros
)

In [16]:
import numpy as np
import pandas as pd

def build_A_meta_constant(years, A):
    """
    Construct A_meta as a dict[year] -> A_y, where A_y is the same A for every year.
    A can be a NumPy array or a pandas DataFrame loaded from CSV.
    """
    years = list(years)
    A_arr = A.to_numpy() if isinstance(A, pd.DataFrame) else np.asarray(A, dtype=float)
    return {y: A_arr for y in years}

In [17]:
import numpy as np
import pandas as pd

def build_f_meta_from_packaging_csv(years, csv_path, n_A):
    """
    Builds f_meta where:
      f[y][0] = Packaging_Plastic_kg for year y
      f[y][i>0] = 0
    """
    df = pd.read_csv(csv_path)

    demand_map = dict(
        zip(df["Year"].astype(int), df["Packaging_Plastic_kg"].astype(float))
    )

    f_meta = {}
    for y in years:
        f = np.zeros(n_A)
        f[0] = demand_map[y]
        f_meta[y] = f

    return f_meta

In [18]:
years = list(range(2025, 2101))

#A meta data
A=[]
A = np.array(A_numeric_df)
n_A = A.shape[0]
# build A_meta (same A for all years)
A_meta = {y: A for y in years}

In [19]:
#f meta data
demand_df = pd.read_csv("data/packaging_production_manual_2025_2100_kg.csv")
demand = dict(zip(demand_df["Year"].astype(int), demand_df["Packaging_Plastic_kg"].astype(float)))
f_meta = {}
for y in years:
    f = np.zeros(n_A)
    f[0] = demand[y]
    f_meta[y] = f

In [20]:
financial_df = pd.read_csv("data/Financial.csv")
A_foreground_df = pd.read_csv("data/Foreground_A.csv")
foreground_s_df = pd.read_csv("data/Foreground Processes Design.csv")
search_elements_foreground = foreground_s_df['provider name'].dropna().astype(str).tolist()
search_elements_finanical = financial_df['LCI Column Index'].dropna().astype(str).tolist()
positive_s_df = pd.read_csv("data/positive_s (1).csv")
search_elements_positive = positive_s_df['provider name'].dropna().astype(str).tolist()
positive_s_indices = [A_design_final_df.iloc[0,:].tolist().index(elem) for elem in search_elements_positive]
positive_s_indices = [i-1 for i in positive_s_indices]

In [72]:
financial_df = pd.read_csv("data/Financial.csv")
financial_df['LCI Column Index'] = financial_df['LCI Column Index'].astype(str)
values_dict = {}
values_dict = (
    financial_df
    .loc[financial_df['LCI Column Index'].isin(search_elements_finanical), ['LCI Column Index', 'Value']]
    .set_index('LCI Column Index')['Value']
    .to_dict()
)
result_dict = {}

# ensure consistent string comparison
A_design_final_df.iloc[:, 0] = A_design_final_df.iloc[:, 0].astype(str)

for i, name in enumerate(A_design_final_df.iloc[:, 0]):
    if name in values_dict:
        result_dict[i - 1] = values_dict[name]

inflation_rate = 0.025
years = range(2025, 2101)

cost_2025 = result_dict  # your existing dictionary

cost_by_year = {
    (i, y): v * (1 + inflation_rate) ** (y - 2025)
    for i, v in cost_2025.items()
    for y in years
}

In [22]:
f_voc_df = pd.read_csv("data/f_voc_every_line_use_collection.csv")
f_voc_df['Process'] = f_voc_df['Process'].astype(str)
values_dict_voc = {}
values_dict_voc = (
    f_voc_df
    .loc[f_voc_df['Process'].isin(search_elements_foreground), ['Process', 'f_VOC_of_total_cost']]
    .set_index('Process')['f_VOC_of_total_cost']
    .to_dict()
)

In [23]:
values_dict_voc

{'Use & Collection': nan,
 'Reclaiming, sorted mixed plastics, PVC Other Food Rigid': 0.7,
 'Polypropylene, PP, virgin resin, non-food bottle grade': 0.75,
 'Plastic Incineration CO2 Capture': 0.5,
 'Plastic Incineration CO2 to the atmosphere': 0.5,
 'Reclaiming, sorted mixed plastics, PET Other Non-food Rigid': 0.7,
 'Microplastics, PP, Marine Environment': 0.9,
 'Polyvinyl chloride resin, suspension grade PVC; non-food grade': 0.75,
 'Microplastics, PP, Filtration': 0.9,
 'Incineration, multi-material food packaging film': 0.5,
 'Landfill, Sanitary (Controlled) - mixed plastic waste': 0.85,
 'Open Burning, multi-material food packaging film': 0.85,
 'Disposal, HDPE Other Food Rigid': 0.85,
 'Reclaiming, Sorted HDPE Other Food Rigid': 0.7,
 'Reclaiming, sorted mixed plastics, PVC Other Non-food Rigid': 0.7,
 'Fermentation CO2 Capture': 0.5,
 'Fermentation CO2 to the atmosphere': 0.55,
 'Disposal, HDPE Other Non-food Rigid': 0.85,
 'Reclaiming, Sorted HDPE Other Non-food Rigid': 0.7,
 

In [70]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import copy
from pyomo.environ import *
from pyomo.environ import RangeSet
from pyomo.environ import value
import plotly.graph_objects as go

In [ ]:
#MODEL
model = ConcreteModel()

#SETS
model.set_N = RangeSet(len(np.transpose(A_design_matrix))) #indices set of all nodes or processes
model.set_T = Set(initialize=range(2025, 2101), ordered=True) #indices set of all years
model.set_Q = Set(initialize=positive_s_indices) #indices set of processes with positive scaling factors
model.set_K = Set(initialize=sorted({i for i, _ in cost_by_year.keys()}),ordered=True)  # index set of background processes used in cost calculations
model.set_L = Set(initialize=foreground_s_indices_columns)

#PARAMETERS
model.unit_cost = Param(model.K, model.T, initialize=cost_by_year, within=Reals)


#VARIABLES
model.s = Var(model.set_N, model.set_T) #scaling factors indexed over processes and time

In [ ]:
def dynamic_roadmap_optimization(
    years, #year vector [2025, 2100]
    A_base_numeric, #Numeric A (np.array) for design
    A_index, #original indices of A
    A_base_full, # full A containing both names and values of rows and columns (for design)
    B_base_numeric, #Numeric B (np.array)
    B_index, #original indices of B
    demand_df, #csv file containing demands
    A_foreground_df, #dataframe containing foreground processes that should be included for cost calculations
    financial_df, #dataframe containing the cost of inputs and outputs
    var_share_cost_df, #share of variable cost in total cost of each process stored in a dataframe
):
    
    
    
    
    

In [18]:
import numpy as np
import pandas as pd
import pyomo.environ as pyo

def minimize_discounted_cost(
    years,      # iterable of years, e.g., range(2025, 2101)
    A_meta,     # dict[year] -> (n_A, n_A) numeric array/DataFrame
    f_meta,     # dict[year] -> (n_A,) numeric array/Series/DataFrame
    n_A,        # number of processes/modules (size of s)
    unit_cost,  # length n_A, in 2025 $ per unit scaling
    discount_rate=0.03,
    solver_name="glpk",
    tee=False,
    nonnegative=True
):
    years = list(years)
    base_year = min(years)

    def to_2d(x):
        if isinstance(x, (pd.DataFrame, pd.Series)):
            x = x.to_numpy()
        x = np.asarray(x, dtype=float)
        if x.ndim == 1:
            x = x.reshape(1, -1)
        return x

    def to_1d(x):
        if isinstance(x, (pd.DataFrame, pd.Series)):
            x = x.to_numpy()
        return np.asarray(x, dtype=float).reshape(-1)

    unit_cost = np.asarray(unit_cost, dtype=float).reshape(-1)

    # build parameter dicts
    A_data, f_data = {}, {}
    for y in years:
        A_y = to_2d(A_meta[y])
        f_y = to_1d(f_meta[y])

        for i in range(n_A):
            f_data[(y, i)] = float(f_y[i])
            for j in range(n_A):
                A_data[(y, i, j)] = float(A_y[i, j])

    m = pyo.ConcreteModel()
    m.Y = pyo.Set(initialize=years, ordered=True)
    m.I = pyo.RangeSet(0, n_A - 1)
    m.J = pyo.RangeSet(0, n_A - 1)

    m.A = pyo.Param(m.Y, m.I, m.J, initialize=A_data, within=pyo.Reals)
    m.f = pyo.Param(m.Y, m.I, initialize=f_data, within=pyo.Reals)

    m.unit_cost = pyo.Param(m.J, initialize={j: float(unit_cost[j]) for j in range(n_A)}, within=pyo.Reals)

    def pv_init(_, y):
        return 1.0 / ((1.0 + float(discount_rate)) ** (int(y) - int(base_year)))
    m.pv = pyo.Param(m.Y, initialize=pv_init, within=pyo.PositiveReals)

    var_domain = pyo.NonNegativeReals if nonnegative else pyo.Reals
    m.s = pyo.Var(m.Y, m.J, domain=var_domain)  # scaling factors by year

    # A[y] s[y] = f[y]
    def a_balance(m, y, i):
        return sum(m.A[y, i, j] * m.s[y, j] for j in m.J) == m.f[y, i]
    m.balance = pyo.Constraint(m.Y, m.I, rule=a_balance)

    # minimize discounted total cost
    def obj(m):
        return sum(m.pv[y] * sum(m.unit_cost[j] * m.s[y, j] for j in m.J) for y in m.Y)
    m.obj = pyo.Objective(rule=obj, sense=pyo.minimize)

    solver = pyo.SolverFactory(solver_name)
    results = solver.solve(m, tee=tee)

    s_df = pd.DataFrame(
        [[pyo.value(m.s[y, j]) for j in range(n_A)] for y in years],
        index=years,
        columns=[f"s_{j}" for j in range(n_A)]
    )

    return s_df, m, results